# Подготовить 5-минутный ролик для просмотра и разметки
В Colab выберите **Среда выполнения → Выполнить все** и разрешите доступ к своему Диску. GPU не нужен. Этот ноутбук читает `yolo/VID_20261118_120325.mp4` и сохраняет **новую** копию `yolo/VID_20261118_120325_review.mp4`: весь проезд в исходном разрешении, 12 кадров/с, без звука, рамок и надписей. Исходник и модель не меняются. Когда появится строка «Готово», сообщите об этом в чате: кадры ещё предстоит посмотреть, разметить и добавить в обучение.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import hashlib
import json
import shutil
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory


def make_review_copy(source, destination, expected_sha256):
    if not source.is_file():
        raise FileNotFoundError(f'Не найден {source}. Укажите путь к папке yolo в следующей ячейке.')
    if destination.exists():
        raise FileExistsError(f'{destination} уже есть. Не перезаписываю; если он готов, сообщите в чат.')
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        raise RuntimeError('Для этой ячейки нужен FFmpeg со ffprobe (есть в обычной среде Colab).')

    def info(path):
        result = subprocess.check_output(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
            '-show_entries', 'stream=width,height,nb_frames:format=duration', '-of', 'json', str(path)])
        data = json.loads(result)
        return data['streams'][0], float(data['format']['duration'])

    with TemporaryDirectory() as temp:
        # Одна последовательная загрузка с Диска; проверяем, что это именно запрошенный проезд.
        local = Path(temp) / 'input.mp4'
        checksum = hashlib.sha256()
        with source.open('rb') as src, local.open('wb') as dst:
            for chunk in iter(lambda: src.read(4 * 1024**2), b''):
                checksum.update(chunk)
                dst.write(chunk)
        if checksum.hexdigest() != expected_sha256:
            raise ValueError('SHA-256 не совпадает с оригиналом. Проверьте путь; результат не сохранён.')
        stream, duration = info(local)
        output = Path(temp) / 'review.mp4'
        subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-xerror', '-i', str(local),
            '-map', '0:v:0', '-an', '-map_metadata', '-1', '-vf', 'setpts=PTS-STARTPTS,fps=12',
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '19', '-maxrate', '2000k',
            '-bufsize', '2000k', '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(output)], check=True)
        review, length = info(output)
        size = output.stat().st_size
        if not (0 < size <= 100 * 1024**2 and abs(length - duration) < 1
                and abs(int(review['nb_frames']) - 12 * duration) < 12
                and sorted((stream['width'], stream['height'])) == sorted((review['width'], review['height']))):
            raise RuntimeError('Размер, разрешение или длительность копии не прошли проверку.')
        with output.open('rb') as src, destination.open('xb') as dst:
            try:
                shutil.copyfileobj(src, dst)
            except BaseException:
                destination.unlink(missing_ok=True)
                raise
    print(f'Готово: {destination} ({size / 1e6:.1f} МБ, {length:.1f} с, {review["nb_frames"]} кадров).')

In [ ]:
folder = Path('/content/drive/MyDrive/yolo')  # измените, если папка лежит не в корне
make_review_copy(folder / 'VID_20261118_120325.mp4', folder / 'VID_20261118_120325_review.mp4',
                 'c218120f3ecb686637b34eed4d6a19cdf13d4f4b767a445ef08bbcd4ae2b346a')